In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi

Thu Sep 10 02:06:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/THU-MIG/yolov10.git
%cd yolov10
!pip install .

Cloning into 'yolov10'...
remote: Enumerating objects: 20338, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 20338 (delta 1), reused 0 (delta 0), pack-reused 20335 (from 3)
Receiving objects: 100% (20338/20338), 11.10 MiB | 13.66 MiB/s, done.
Resolving deltas: 100% (14353/14353), done.
/content/yolov10
Processing /content/yolov10
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.1.34-py3-none-any.whl size=732491 sha256=b4285b379fa6040113f5acbd7b629e251d1e82b8ae6ed867cefbff597c2d3d98
  Stored in directory: /tmp/pip-ephem-wheel-cache-ffw9t5cv/wheels/8a/d7/2b/a0e00d40ee1d82400823d08e7462237ae204351f09013ff66e
Successfully built ultralytics


In [4]:
!pip install albumentations==1.4 #PARA FAZER AUMENTO DE DADOS.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: albumentations
    Found existing installation: albumentations 2.0.8
    Uninstalling albumentations-2.0.8:
      Successfully uninstalled albumentations-2.0.8


In [5]:
!wget -O /content/yolov10n.pt \
https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10n.pt

--2026-09-10 02:06:55--  https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10n.pt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/804788522/411e0d4f-1023-40ad-bfdd-c99f0dddb73b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-10T02%3A49%3A12Z&rscd=attachment%3B+filename%3Dyolov10n.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-09-10T01%3A49%3A04Z&ske=2026-09-10T02%3A49%3A12Z&sks=b&skv=2018-11-09&sig=wW86%2F58U3pRX6gLOFLQa0c10B6DrSmNnAwuWRCz9BPI%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4OTAwNzgxNSwibmJmIjoxNzg5MDA2MDE1LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmUud2

In [6]:
import torch
from ultralytics.nn.tasks import YOLOv10DetectionModel

torch.serialization.add_safe_globals([YOLOv10DetectionModel])

In [7]:
import torch

_original_torch_load = torch.load

def torch_load_compat(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)

torch.load = torch_load_compat

In [8]:
from ultralytics import YOLO

model = YOLO("/content/yolov10n.pt")

print("✅ YOLOv10 carregado!")


config = "/content/drive/MyDrive/Trainv10/config.yaml"

results = model.train(
    data=config,
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    flipud=1.0,
    device=0
)

✅ YOLOv10 carregado!
New https://pypi.org/project/ultralytics/8.4.146 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.34 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/content/yolov10n.pt, data=/content/drive/MyDrive/Trainv10/config.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, val_period=1, cache=False, device=0, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, cl

100%|██████████| 755k/755k [00:00<00:00, 21.0MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1      9856  ultralytics.nn.modules.block.SCDown          [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1     36096  ultralytics

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


...

                   all        348        204      0.109      0.157     0.0409     0.0124



100 epochs completed in 2.793 hours.
Optimizer stripped from /content/yolov10/runs/detect/train/weights/last.pt, 5.7MB
Optimizer stripped from /content/yolov10/runs/detect/train/weights/best.pt, 5.7MB

Validating /content/yolov10/runs/detect/train/weights/best.pt...
Ultralytics YOLOv8.1.34 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10n summary (fused): 285 layers, 2694806 parameters, 0 gradients, 8.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [00:05<00:00,  1.92it/s]


                   all        348        204      0.332     0.0925     0.0913     0.0301
Speed: 0.3ms preprocess, 4.3ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /content/yolov10/runs/detect/train


lr/pg0,█▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr/pg1,████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
lr/pg2,▆████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
metrics/mAP50(B),▁▁▁▁▁▁▁▁▂▃▅▂▃▃▃▃▃▅▄▄▇█▄▇▆▅▄▄▄▄▇▅▅▅▄▅▆▅▅▄
metrics/mAP50-95(B),▁▁▁▁▁▂▁▂▄▂▂▃▃▃▄▄▄▅▆▆█▇▅▄▃▃▄▄▄▄▄▄▅▅▅▄▄▅▄▄
metrics/precision(B),▁▁▁▁▁▁▁▁▁▁▁▂▁▃▂▁▁▆▅▂▄▃▄█▃▃▄▅▅▅▅▃▂▃▃▂▃▃▄▇
metrics/recall(B),▃▁▇▇▇▇█▁▇█▁▁█▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+12,...


In [9]:
import shutil

diretorio_origem = '/content/yolov10/runs/detect/train'
diretorio_destino = '/content/drive/MyDrive/Trainv10/model_n_100' #cria um novo diretorio no drive.

shutil.copytree(diretorio_origem, diretorio_destino)

'/content/drive/MyDrive/Trainv10/model_n_100'